# Import bibliotek

In [1]:
import pandas as pd
import requests

## Load data and aggregate

In [2]:
df = pd.read_csv("data\\5-top_1000_artykulow_monthly.csv")

In [3]:
df = df.sort_values(by=["page_id","year","month"])


In [68]:
df_rank = df[["title","views"]].groupby("title").sum("views").sort_values("views", ascending=False)
df_top10 = df_rank.head(10)

### Weryfikacja Pustych page_id
Puste pageid pochodzą ze stron które są usunięte lub zostały przeniesione do innych

In [50]:
df_blankId = df[df["page_id"].isna()][["page_id","title","views"]]

In [51]:
df_blankId = df_blankId.groupby(by='title').sum('views')

### Weryfikacja ostatniego tytułu 
Po wykonaniu poniższego kodu wyszło, że w ramach danego page_id występuje ten sam tytuł

In [ ]:
last_titles = df.groupby("page_id")["title"].last()

df["last_title"] = df["page_id"].map(last_titles)

df_changed = df[df["last_title"]!=df["title"]]

#sprawdzenie czy istnieją różne tytuły dla tego page_id
df.groupby("page_id")["title"].nunique().sort_values()


## Search categories via API

In [61]:
def search_category(article_name):
        '''
        Wyszukiwanie tagow z last fm api
        '''

        headers = {
            "User-Agent" : "MarcinBot/1.0"
        }

        params = {
                "action": "query",
                "prop": "categories",
                "titles": article_name,
                "cllimit": "max",
                "format": "json"
            }

        response = requests.get("https://pl.wikipedia.org/w/api.php", params=params, headers=headers)

        if response.status_code == 200:
            data = response.json()
            pages = data["query"]["pages"]
            rows = []

            for pid, page in pages.items():
                for cat in page.get("categories", []):
                    rows.append({
                        "pageid": pid,
                        "title": page["title"],
                        "category": cat["title"]
                    })

            df_categories = pd.DataFrame(rows)
            print(df_categories)
        else:
            tag_list = []
            print(f"Błąd: {response.status_code}")

        return df_categories

In [66]:
df_cat = search_category("Polska")

   pageid   title                                           category
0   44895  Polska  Kategoria:Artykuły, które powinna mieć każda W...
1   44895  Polska        Kategoria:Artykuły z propozycjami tłumaczeń
2   44895  Polska  Kategoria:Członkowie Organizacji Narodów Zjedn...
3   44895  Polska          Kategoria:Hasła kanonu polskiej Wikipedii
4   44895  Polska   Kategoria:Państwa członkowskie Unii Europejskiej
5   44895  Polska                 Kategoria:Państwa należące do NATO
6   44895  Polska              Kategoria:Państwa w Europie Środkowej
7   44895  Polska                                   Kategoria:Polska
8   44895  Polska       Kategoria:Strony korzystające z podprzypisów
9   44895  Polska    Kategoria:Szablon cytowania używa pól opisowych
10  44895  Polska             Kategoria:Szablon cytuj do sprawdzenia
11  44895  Polska  Kategoria:Szablony cytowania – problemy – cytu...


In [86]:
for _, row in df_top10.iterrows():
    title = row.name
    df_cat = search_category(title)
    print(f"Search category: {title}")

    # zakładam, że df_cat ma kolumny: title, category
    for _, cat_row in df_cat.iterrows():
        rows.append({
            "title": title,
            "category": cat_row["category"],
            "pageid": cat_row["pageid"]
        })

df_all_categories = pd.DataFrame(rows)

df_merged = df_top10.merge(df_all_categories, on="title", how="left")

   pageid   title                                           category
0   44895  Polska  Kategoria:Artykuły, które powinna mieć każda W...
1   44895  Polska        Kategoria:Artykuły z propozycjami tłumaczeń
2   44895  Polska  Kategoria:Członkowie Organizacji Narodów Zjedn...
3   44895  Polska          Kategoria:Hasła kanonu polskiej Wikipedii
4   44895  Polska   Kategoria:Państwa członkowskie Unii Europejskiej
5   44895  Polska                 Kategoria:Państwa należące do NATO
6   44895  Polska              Kategoria:Państwa w Europie Środkowej
7   44895  Polska                                   Kategoria:Polska
8   44895  Polska       Kategoria:Strony korzystające z podprzypisów
9   44895  Polska    Kategoria:Szablon cytowania używa pól opisowych
10  44895  Polska             Kategoria:Szablon cytuj do sprawdzenia
11  44895  Polska  Kategoria:Szablony cytowania – problemy – cytu...
Search category: Polska
  pageid   title                                           category
0    639  B

UnboundLocalError: cannot access local variable 'df_categories' where it is not associated with a value

In [89]:
df_merged.to_csv("data\\top10_with_categories.csv", index=False)

In [56]:
# params = {
#     'action': "query",
#     'prop' : 'categories',
#     'titles': "Poznań",
#     'format' : 'json'
# }
headers = {
    "User-Agent" : "MarcinBot/1.0"
}

params = {
        "action": "query",
        "prop": "categories",
        "titles": "Polska",
        "cllimit": "max",
        "format": "json"
    }

response = requests.get("https://pl.wikipedia.org/w/api.php", params=params, headers=headers)


if response.status_code == 200:
    data = response.json()

else:
    print(f"Błąd: {response.status_code}")

In [60]:
pages = data["query"]["pages"]
rows = []

for pid, page in pages.items():
    for cat in page.get("categories", []):
        rows.append({
            "pageid": pid,
            "title": page["title"],
            "category": cat["title"]
        })

df_categories = pd.DataFrame(rows)
print(df_categories)

   pageid   title                                           category
0   44895  Polska  Kategoria:Artykuły, które powinna mieć każda W...
1   44895  Polska        Kategoria:Artykuły z propozycjami tłumaczeń
2   44895  Polska  Kategoria:Członkowie Organizacji Narodów Zjedn...
3   44895  Polska          Kategoria:Hasła kanonu polskiej Wikipedii
4   44895  Polska   Kategoria:Państwa członkowskie Unii Europejskiej
5   44895  Polska                 Kategoria:Państwa należące do NATO
6   44895  Polska              Kategoria:Państwa w Europie Środkowej
7   44895  Polska                                   Kategoria:Polska
8   44895  Polska       Kategoria:Strony korzystające z podprzypisów
9   44895  Polska    Kategoria:Szablon cytowania używa pól opisowych
10  44895  Polska             Kategoria:Szablon cytuj do sprawdzenia
11  44895  Polska  Kategoria:Szablony cytowania – problemy – cytu...


## Wylistowanie kategorii 

In [100]:
df_rank = df_rank[['views']].head(100)

In [101]:
df_rank.to_csv("data//top100.csv")

In [96]:
suma = df_rank["views"].sum()
print (suma*0.8)
print (suma)

3095237933.6000004
3869047417


## ranking kategorii
Przydzielone przez CoPilot

In [102]:
df_categories = pd.read_csv("data//top100_cat.csv")

In [103]:
df_top_cat = df_categories.groupby("assigned_category").sum("views")